# Multi-Dataset Tokenization A/B Study with LLaVA-1.5 (ALL ENTITIES VERSION)

This notebook:

- Loads LLaVA-1.5 (Hugging Face) via the repo's VLM loader
- Samples from TallyQA, RefCOCO, and TextVQA datasets
- Identifies ALL noun/adjective/adverb entities in each question/expression via POS tagging (spaCy)
- Enforces minimal interventions that change ALL eligible entities from 1 token (A) to 2 tokens (B)
- Evaluates each intervention separately across all datasets to measure A vs B performance disparity
- Verifies that interventions actually flip tokenization as intended

Key requirement: For each question/expression, ALL noun/adjective/adverb entities that can be converted from 1→2 tokens are modified simultaneously using the same intervention type.

## Datasets:
- **TallyQA**: Counting-focused VQA with simple/complex splits
- **RefCOCO**: Referring expression grounding (bounding box prediction)
- **TextVQA**: VQA requiring reading text in images


In [1]:
# Install spaCy English model if needed
import sys
import subprocess

def pip_install(pkg: str) -> None:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

try:
    import spacy  # type: ignore
except Exception:
    pip_install("spacy==3.7.3")
    import spacy  # type: ignore

try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    pip_install(
        "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
    )
    import en_core_web_sm  # type: ignore

    nlp = en_core_web_sm.load()

print("spaCy model loaded:", nlp)


spaCy model loaded: <spacy.lang.en.English object at 0x7f4be841d630>


In [2]:
# Imports from this repo and base libs
import json
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
from PIL import Image

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Path configuration
DATA_ROOT = Path("/localdisk/ssrivas9/vlm-evaluation")

# Dataset metadata paths
TALLYQA_META = DATA_ROOT / "datasets/tally-qa/metadata-slim-1024.json"
REFCOCO_META = DATA_ROOT / "datasets/refcoco/metadata-slim-1024.json" 
TEXTVQA_META = DATA_ROOT / "datasets/text-vqa/metadata-slim-1024.json"
VQAV2_META = DATA_ROOT / "datasets/vqa-v2/metadata-slim-1024.json"

# HF token handling: either env var or .hf_token file in repo root
HF_TOKEN = None
if (DATA_ROOT / ".hf_token").exists():
    HF_TOKEN = (DATA_ROOT / ".hf_token").read_text().strip()
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

print("Using HF token:", "yes" if HF_TOKEN else "no")

print(f"Data root: {DATA_ROOT}")
print(f"TallyQA metadata: {TALLYQA_META}")
print(f"RefCOCO metadata: {REFCOCO_META}")
print(f"TextVQA metadata: {TEXTVQA_META}")
print(f"VQAv2 metadata: {VQAV2_META}")


Using HF token: yes
Data root: /localdisk/ssrivas9/vlm-evaluation
TallyQA metadata: /localdisk/ssrivas9/vlm-evaluation/datasets/tally-qa/metadata-slim-1024.json
RefCOCO metadata: /localdisk/ssrivas9/vlm-evaluation/datasets/refcoco/metadata-slim-1024.json
TextVQA metadata: /localdisk/ssrivas9/vlm-evaluation/datasets/text-vqa/metadata-slim-1024.json
VQAv2 metadata: /localdisk/ssrivas9/vlm-evaluation/datasets/vqa-v2/metadata-slim-1024.json


In [3]:
# Load dataset index files and create unified dataset structure
from vlm_eval.tasks.harnesses.tallyqa import TallyQAIndexDataset
from vlm_eval.tasks.harnesses.refcoco import RefCOCOIndexDataset  
from vlm_eval.tasks.harnesses.textvqa import TextVQAIndexDataset
from vlm_eval.tasks.harnesses.vqav2 import VQAv2IndexDataset
import json
from pathlib import Path
from typing import Tuple

# Create a custom TextVQA dataset that handles the answers field correctly
class FixedTextVQAIndexDataset:
    def __init__(self, root_dir: Path, index_file: Path):
        self.root_dir, self.index_file = root_dir, index_file
        with open(self.root_dir / self.index_file, "r") as f:
            self.examples = list(json.load(f).values())

    def __getitem__(self, idx: int) -> Tuple[int, str, Path, list]:
        """Return (question_id: int, question: str, img_path: Path, answers: list) for an example."""
        ex = self.examples[idx]
        return ex["question_id"], ex["question"], Path(self.root_dir / ex["img_path"]), ex["answers"]

    def __len__(self) -> int:
        return len(self.examples)

# Load individual datasets
tallyqa_dataset = TallyQAIndexDataset(DATA_ROOT, TALLYQA_META.relative_to(DATA_ROOT))
refcoco_dataset = RefCOCOIndexDataset(DATA_ROOT, REFCOCO_META.relative_to(DATA_ROOT))
textvqa_dataset = FixedTextVQAIndexDataset(DATA_ROOT, TEXTVQA_META.relative_to(DATA_ROOT))
vqav2_dataset = VQAv2IndexDataset(DATA_ROOT, VQAV2_META.relative_to(DATA_ROOT))

print(f"TallyQA: {len(tallyqa_dataset)} examples")
print(f"RefCOCO: {len(refcoco_dataset)} examples")
print(f"TextVQA: {len(textvqa_dataset)} examples")
print(f"VQAv2: {len(vqav2_dataset)} examples")

# Create unified dataset structure
class UnifiedDataset:
    def __init__(self):
        self.datasets = {
            'tallyqa': tallyqa_dataset,
            'refcoco': refcoco_dataset,
            'textvqa': textvqa_dataset,
            'vqav2': vqav2_dataset
        }
        
    def get_example(self, dataset_name: str, idx: int) -> Dict[str, Any]:
        """Get example in unified format: {id, text, img_path, answer, dataset}"""
        if dataset_name == 'tallyqa':
            qid, question, img_path, answer = self.datasets[dataset_name][idx]
            return {
                'id': qid,
                'text': question,
                'img_path': img_path,
                'answer': str(answer),
                'dataset': 'tallyqa'
            }
        elif dataset_name == 'refcoco':
            ex_id, ref_expr, img_path, bbox = self.datasets[dataset_name][idx]
            return {
                'id': ex_id,
                'text': ref_expr,
                'img_path': img_path,
                'answer': str(bbox.tolist()),  # bbox as string
                'dataset': 'refcoco'
            }
        elif dataset_name == 'textvqa':
            qid, question, img_path, answers = self.datasets[dataset_name][idx]
            return {
                'id': qid,
                'text': question,
                'img_path': img_path,
                'answer': answers,  # Keep as list for multiple valid answers
                'dataset': 'textvqa'
            }
        elif dataset_name == 'vqav2':
            qid, question, img_path, answer = self.datasets[dataset_name][idx]
            return {
                'id': qid,
                'text': question,
                'img_path': img_path,
                'answer': str(answer),  # VQAv2 answer as string
                'dataset': 'vqav2'
            }
        else:
            raise ValueError(f"Unknown dataset: {dataset_name}")
    
    def get_dataset_size(self, dataset_name: str) -> int:
        return len(self.datasets[dataset_name])

unified_dataset = UnifiedDataset()

# Test unified interface
for dataset_name in ['tallyqa', 'refcoco', 'textvqa', 'vqav2']:
    example = unified_dataset.get_example(dataset_name, 0)
    print(f"\n{dataset_name.upper()} example:")
    print(f"  ID: {example['id']}")
    print(f"  Text: {example['text'][:100]}...")
    print(f"  Answer: {example['answer'][:50]}...")


TallyQA: 2048 examples
RefCOCO: 3072 examples
TextVQA: 1024 examples
VQAv2: 1024 examples

TALLYQA example:
  ID: 30108301
  Text: How many planes?...
  Answer: 1...

REFCOCO example:
  ID: 5553170074492262299
  Text: elephant on the left behind tree...
  Answer: [0.01, 0.01, 0.49, 0.75]...

TEXTVQA example:
  ID: 36937
  Text: This is book material?
Reference OCR token: BEETHOVEN, WING, EMPEROR, piano, concerto, NIKITA, MAGAL...
  Answer: ['answering does not require reading text in the image', 'unanswerable', 'no', 'yes', 'no', 'unanswerable', 'no', 'no', 'answering does not require reading text in the image', 'not a question']...

VQAV2 example:
  ID: 422700016
  Text: What is the boy listening to?...
  Answer: parents...


In [4]:
# Load LLaVA-1.5 model using repo loader
# We use the official HF hub id via the model family 'llava-v15'
from vlm_eval.models import load_vlm

MODEL_FAMILY = "llava-v15"
MODEL_ID = "llava-v1.5-7b"
RUN_DIR = Path("liuhaotian/llava-v1.5-7b")  # hf hub path is accepted by loader

vlm = load_vlm(
    model_family=MODEL_FAMILY,
    model_id=MODEL_ID,
    run_dir=RUN_DIR,
    hf_token=HF_TOKEN,
    load_precision="bf16",
    max_length=128,
    temperature=0.2,
)
tokenizer = vlm.tokenizer  # Access tokenizer directly
image_processor = vlm.image_processor

print("Loaded VLM:", MODEL_ID)

# Get prompt functions for each dataset
# Use the base dataset IDs, not the slim variants
tallyqa_prompt_fn = vlm.get_prompt_fn("tally-qa")
refcoco_prompt_fn = vlm.get_prompt_fn("refcoco")
textvqa_prompt_fn = vlm.get_prompt_fn("text-vqa")
vqav2_prompt_fn = vlm.get_prompt_fn("vqa-v2")

prompt_fns = {
    'tallyqa': tallyqa_prompt_fn,
    'refcoco': refcoco_prompt_fn,
    'textvqa': textvqa_prompt_fn,
    'vqav2': vqav2_prompt_fn
}

print("\nPrompt function examples:")
for dataset_name in ['tallyqa', 'refcoco', 'textvqa']:
    example = unified_dataset.get_example(dataset_name, 0)
    if dataset_name == 'tallyqa':
        prompt = prompt_fns[dataset_name](example['text'], choices=[str(i) for i in range(16)])
    else:
        prompt = prompt_fns[dataset_name](example['text'])
    print(f"\n{dataset_name.upper()} prompt: {prompt}...")


/localdisk/ssrivas9/miniconda3/envs/eval/lib/python3.10/site-packages/prismatic/models/load.py


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loaded llava llava-v1.5-7b
Loaded VLM: llava-v1.5-7b

Prompt function examples:

TALLYQA prompt: A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>
How many planes?
A. 0
B. 1
C. 2
D. 3
E. 4
F. 5
G. 6
H. 7
I. 8
J. 9
K. 10
L. 11
M. 12
N. 13
O. 14
P. 15
Answer with the option's letter from the given choices directly. ASSISTANT:...

REFCOCO prompt: A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>
Please provide the bounding box coordinate of the region this sentence describes: "elephant on the left behind tree" ASSISTANT:...

TEXTVQA prompt: A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>
This is book material?
Reference OCR token: BEETH

In [5]:
# Helper functions for entity detection and tokenization
from transformers import PreTrainedTokenizerBase, AutoTokenizer
from spacy.matcher import Matcher

# Create spaCy matcher for noun phrases
matcher = Matcher(nlp.vocab)
matcher.add("NOUN_PHRASE", [[{"POS": "DET", "OP": "?"}, {"POS": "ADJ", "OP": "*"}, {"POS": "NOUN", "OP": "+"}]])

def find_candidate_entities(text: str, max_len: int = 3) -> List[str]:
    """Find noun/adjective/adverb entities in text using spaCy."""
    doc = nlp(text)
    spans = []
    
    # Use noun chunks first
    for chunk in doc.noun_chunks:
        t = chunk.text.strip()
        if 1 <= len(t.split()) <= max_len:
            spans.append(t)
    
    # Also try simple matcher
    for match_id, start, end in matcher(doc):
        t = doc[start:end].text.strip()
        if 1 <= len(t.split()) <= max_len:
            spans.append(t)
    
    # Add individual nouns, adjectives, adverbs
    for token in doc:
        if token.pos_ in {"NOUN", "PROPN", "PRON", "ADJ", "ADV"} and len(token.text.strip()) > 2:
            spans.append(token.text)
    
    # Deduplicate with order preserved
    seen = set()
    uniq = []
    for s in spans:
        if s.lower() not in seen:
            seen.add(s.lower())
            uniq.append(s)
    return uniq

def tokenized_length(s: str) -> int:
    """Return number of tokens for given string."""
    return len(tokenizer.encode(s, add_special_tokens=False))

# Try to get fast tokenizer for offset mapping (fallback gracefully if unavailable)
try:
    tokenizer_fast = AutoTokenizer.from_pretrained(str(RUN_DIR), use_fast=True)
    print("Fast tokenizer available for offset mapping")
except Exception as e:
    tokenizer_fast = None
    print("Fast tokenizer unavailable (will use approximations):", e)

# Test entity detection
test_texts = [
    "How many red cars are in the image?",
    "Point to the large brown dog sitting near the tree",
    "What text is written on the blue sign?"
]

print("\nEntity detection examples:")
for i, text in enumerate(test_texts):
    entities = find_candidate_entities(text)
    print(f"Text {i+1}: {text}")
    print(f"  Entities: {entities}")
    print(f"  Token lengths: {[tokenized_length(e) for e in entities]}")

print(f"\nConfiguration: Targeting 100 examples per dataset (400 total per intervention)")
print("Available dataset sizes: TallyQA=2048, RefCOCO=3072, TextVQA=1024, VQAv2=1024")


Fast tokenizer available for offset mapping

Entity detection examples:
Text 1: How many red cars are in the image?
  Entities: ['the image', 'many red cars', 'red cars', 'cars', 'image', 'many', 'red']
  Token lengths: [2, 3, 2, 1, 1, 1, 1]
Text 2: Point to the large brown dog sitting near the tree
  Entities: ['the tree', 'large brown dog', 'brown dog', 'dog', 'tree', 'large', 'brown']
  Token lengths: [2, 3, 2, 1, 1, 1, 1]
Text 3: What text is written on the blue sign?
  Entities: ['What text', 'the blue sign', 'text', 'blue sign', 'sign', 'blue']
  Token lengths: [2, 3, 1, 2, 1, 1]

Configuration: Targeting 100 examples per dataset (400 total per intervention)
Available dataset sizes: TallyQA=2048, RefCOCO=3072, TextVQA=1024, VQAv2=1024


In [6]:
# Build A/B with ALL entities via minimal, local prompt edits

# Define intervention transforms
TRANSFORMS = [
    ("wrap_quotes", '"', '"'),
    ("wrap_parentheses", "(", ")"),
    ("wrap_brackets", "[", "]"),
    ("wrap_unicode_quotes", "\u201c", "\u201d"),  # Smart quotes
    ("prefix_space", " ", ""),
    ("prefix_underscore", "_", ""),
]

def get_offsets(text: str):
    """Get offset mappings and token IDs from fast tokenizer if available."""
    if tokenizer_fast is None:
        return None, None
    try:
        encoded = tokenizer_fast.encode_plus(text, return_offsets_mapping=True, add_special_tokens=False)
        return encoded["offset_mapping"], encoded["input_ids"]
    except NotImplementedError:
        return None, None

def entity_token_len_in_text(text: str, start: int, length: int) -> Optional[int]:
    """Calculate exact token count of entity within text using character offsets."""
    offsets, token_ids = get_offsets(text)
    if offsets is None or token_ids is None:
        return None
    
    end = start + length
    start_token = end_token = None
    
    for i, (char_start, char_end) in enumerate(offsets):
        if char_start <= start < char_end and start_token is None:
            start_token = i
        if char_start < end <= char_end:
            end_token = i + 1
            break
    
    return (end_token - start_token) if start_token is not None and end_token is not None else None

def find_all_occurrences(text: str, sub: str) -> List[int]:
    """Find all occurrences of substring in text."""
    positions = []
    start = 0
    while True:
        pos = text.lower().find(sub.lower(), start)
        if pos == -1:
            break
        positions.append(pos)
        start = pos + 1
    return positions

def build_wrapped_variant(q: str, ent: str, pos: int, pre: str, post: str) -> Tuple[str, int]:
    """Build new question with entity wrapped at given position."""
    wrapped = pre + ent + post
    new_q = q[:pos] + wrapped + q[pos + len(ent):]
    new_pos = pos + len(pre)  # Position of entity within wrapped version
    return new_q, new_pos

def select_all_entities_local_edits(text: str, dataset_name: str) -> Dict[str, List[Dict[str, Any]]]:
    """Return dict: intervention_name -> list of text variants where ALL eligible entities are wrapped."""
    buckets: Dict[str, List[Dict[str, Any]]] = {name: [] for name, _, _ in TRANSFORMS}
    cands = find_candidate_entities(text)
    
    # For each intervention type, try to convert ALL eligible entities
    for intervention_name, pre, post in TRANSFORMS:
        eligible_entities = []
        validation_details = []
        
        # Find all entities that are 1-token in isolation
        for ent in cands:
            # Check if entity is 1-token in isolation (simplified check)
            isolated_tokens = tokenized_length(ent)
            if isolated_tokens == 1:
                # Check if wrapping changes tokenization in isolation
                wrapped_ent = pre + ent + post
                wrapped_tokens = tokenized_length(wrapped_ent)
                
                # If wrapping increases token count, it's a candidate
                if wrapped_tokens > isolated_tokens:
                    occs = find_all_occurrences(text, ent)
                    for pos in occs:
                        eligible_entities.append((ent, pos))
                        validation_details.append({
                            'entity': ent,
                            'isolated_tokens': isolated_tokens,
                            'wrapped_tokens': wrapped_tokens,
                            'position': pos
                        })
        
        # If we have eligible entities, create a version with ALL of them converted
        if eligible_entities:
            # Remove duplicates while preserving order
            seen_entities = set()
            unique_entities = []
            for ent, pos in eligible_entities:
                if ent not in seen_entities:
                    seen_entities.add(ent)
                    unique_entities.append((ent, pos))
            
            # Sort by position in reverse order to avoid position shifts during replacement
            unique_entities.sort(key=lambda x: x[1], reverse=True)
            
            qB = text
            converted_entities = []
            
            # Apply intervention to all unique eligible entities
            for ent, pos in unique_entities:
                # Find current position (may have shifted due to previous edits)
                current_pos = qB.lower().find(ent.lower())
                if current_pos != -1:
                    qB, _ = build_wrapped_variant(qB, ent, current_pos, pre, post)
                    converted_entities.append(ent)
            
            if converted_entities:
                buckets[intervention_name].append({
                    "entities": converted_entities,
                    "entity_count": len(converted_entities), 
                    "textA": text, 
                    "textB": qB,
                    "intervention": intervention_name,
                    "dataset": dataset_name,
                    "validation_details": validation_details[:5]  # Keep first 5 for debugging
                })
    
    return buckets

# Test intervention logic
print("Testing intervention logic:")
test_text = "How many red cars are in the parking lot?"
results = select_all_entities_local_edits(test_text, "test")

for intervention, items in results.items():
    if items:
        item = items[0]
        print(f"\n{intervention}:")
        print(f"  Entities: {item['entities']}")
        print(f"  A: {item['textA']}")
        print(f"  B: {item['textB']}")


Testing intervention logic:

wrap_quotes:
  Entities: ['lot', 'cars', 'red', 'many']
  A: How many red cars are in the parking lot?
  B: How "many" "red" "cars" are in the parking "lot"?

wrap_parentheses:
  Entities: ['lot', 'cars', 'red', 'many']
  A: How many red cars are in the parking lot?
  B: How (many) (red) (cars) are in the parking (lot)?

wrap_brackets:
  Entities: ['lot', 'cars', 'red', 'many']
  A: How many red cars are in the parking lot?
  B: How [many] [red] [cars] are in the parking [lot]?

wrap_unicode_quotes:
  Entities: ['lot', 'cars', 'red', 'many']
  A: How many red cars are in the parking lot?
  B: How “many” “red” “cars” are in the parking “lot”?

prefix_space:
  Entities: ['lot', 'cars', 'red', 'many']
  A: How many red cars are in the parking lot?
  B: How  many  red  cars are in the parking  lot?

prefix_underscore:
  Entities: ['lot', 'cars', 'red', 'many']
  A: How many red cars are in the parking lot?
  B: How _many _red _cars are in the parking _lot?


In [7]:
# Aggregate examples per intervention across all datasets
per_intervention: Dict[str, List[Dict[str, Any]]] = {name: [] for name, _, _ in TRANSFORMS}

# Process each dataset
dataset_configs = {
    'tallyqa': {'target': 100, 'processed': 0},
    'refcoco': {'target': 100, 'processed': 0}, 
    'textvqa': {'target': 100, 'processed': 0},
    'vqav2': {'target': 100, 'processed': 0}
}

print("Collecting examples from all datasets...")

for dataset_name in ['tallyqa', 'refcoco', 'textvqa', 'vqav2']:
    print(f"\nProcessing {dataset_name}...")
    dataset_size = unified_dataset.get_dataset_size(dataset_name)
    processing_errors = 0
    
    for i in range(min(dataset_size, 1000)):  # Limit search to first 1000 examples
        if i % 100 == 0:
            print(f"  Processed {i}/{dataset_size} examples (errors: {processing_errors})")
            
        try:
            example = unified_dataset.get_example(dataset_name, i)
            
            # Validate example structure
            required_keys = ['id', 'text', 'img_path', 'answer', 'dataset']
            missing_keys = [key for key in required_keys if key not in example]
            if missing_keys:
                print(f"  Warning: Example {i} missing keys: {missing_keys}")
                processing_errors += 1
                continue
            
            # Log TallyQA ground truth format for debugging
            if dataset_name == 'tallyqa' and i < 5:
                print(f"  TallyQA GT sample {i}: {example['answer']} (type: {type(example['answer'])})")
            
            found = select_all_entities_local_edits(example['text'], dataset_name)
            
        except Exception as e:
            print(f"  Error processing example {i} from {dataset_name}: {e}")
            processing_errors += 1
            continue
        
        for intervention_name in per_intervention.keys():
            for item in found.get(intervention_name, []):
                # Add tokenization logging
                textA = item["textA"]
                textB = item["textB"]
                entity = item["entities"][0] if item["entities"] else "unknown"
                store_tokenization_info(item, textA, textB, entity, intervention_name)
                
                # Log first few examples
                if len(per_intervention[intervention_name]) < 3:
                    log_tokenization_comparison(textA, textB, entity, intervention_name)
                
                per_intervention[intervention_name].append({
                    "id": example['id'],
                    "original_text": example['text'],
                    "img_path": example['img_path'],
                    "answer": example['answer'],
                    "dataset": dataset_name,
                    **item,
                })
        
        # Check if we have enough examples per dataset for each intervention
        dataset_counts = {}
        for intervention_name in per_intervention.keys():
            count = sum(1 for item in per_intervention[intervention_name] if item['dataset'] == dataset_name)
            dataset_counts[intervention_name] = count
        
        # Stop if we have enough examples from this dataset
        if all(count >= dataset_configs[dataset_name]['target'] for count in dataset_counts.values()):
            print(f"  Sufficient examples found for {dataset_name}")
            break

# Report counts per intervention and dataset
print("\nExample counts per intervention and dataset:")
for intervention_name, items in per_intervention.items():
    dataset_breakdown = {}
    for item in items:
        dataset = item['dataset']
        dataset_breakdown[dataset] = dataset_breakdown.get(dataset, 0) + 1
    
    total = len(items)
    print(f"{intervention_name}: {total} total ({dataset_breakdown})")

# Show example A/B pairs with ALL entities modified
print("\nExample A/B pairs with ALL entities modified:")
for intervention_name, items in per_intervention.items():
    if items:
        # Show one example from each dataset
        dataset_examples = {}
        # Pick a different item for each intervention by using a different index for each intervention
        # We'll use the hash of the intervention name to select a different item per intervention per dataset
        for idx, item in enumerate(items):
            dataset = item['dataset']
            if dataset not in dataset_examples:
                # Use a deterministic but different index for each intervention
                intervention_offset = abs(hash(intervention_name)) % len(items)
                chosen_item = items[(idx + intervention_offset) % len(items)]
                dataset_examples[dataset] = chosen_item
                if len(dataset_examples) == len(set([it['dataset'] for it in items])):
                    break
        
        print(f"\n{intervention_name}:")
        for dataset, ex in dataset_examples.items():
            print(f"  {dataset.upper()} - Entities ({ex['entity_count']}): {ex['entities'][:3]}{'...' if len(ex['entities']) > 3 else ''}")
            print(f"    A: {ex['textA'][:80]}...")
            print(f"    B: {ex['textB'][:80]}...")



Processing tallyqa...
  Processed 0/2048 examples (errors: 0)
  TallyQA GT sample 0: 1 (type: <class 'str'>)


NameError: name 'store_tokenization_info' is not defined

In [ ]:
# Enhanced tokenization logging and analysis functions
def get_tokenization_details(text: str) -> dict:
    """Get detailed tokenization information for a text string."""
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    tokens_decoded = [tokenizer.decode([token_id]) for token_id in token_ids]
    reconstructed = tokenizer.decode(token_ids, skip_special_tokens=True)
    
    return {
        "text": text,
        "token_count": len(token_ids),
        "token_ids": token_ids,
        "tokens_decoded": tokens_decoded,
        "reconstructed": reconstructed,
        "matches_original": text == reconstructed
    }

def log_tokenization_comparison(textA: str, textB: str, entity: str, intervention: str):
    """Log detailed tokenization comparison between A and B texts."""
    detailsA = get_tokenization_details(textA)
    detailsB = get_tokenization_details(textB)
    
    print(f"\n--- TOKENIZATION COMPARISON: {intervention.upper()} ---")
    print(f"Entity: '{entity}'")
    print(f"Original (A): '{textA}'")
    print(f"Modified (B): '{textB}'")
    print(f"Token change: +{detailsB["token_count"] - detailsA["token_count"]}")
    
    print(f"\nDetailed breakdown:")
    print(f"  Original tokens ({detailsA["token_count"]}):")
    for i, token in enumerate(detailsA["tokens_decoded"]):
        print(f"    [{i}] '{token}'")
    
    print(f"  Modified tokens ({detailsB["token_count"]}):")
    for i, token in enumerate(detailsB["tokens_decoded"]):
        print(f"    [{i}] '{token}'")
    
    print(f"\nReconstruction verification:")
    print(f"  Original matches: {detailsA["matches_original"]}")
    print(f"  Modified matches: {detailsB["matches_original"]}")
    if not detailsA["matches_original"]:
        print(f"    Original reconstructed: '{detailsA["reconstructed"]}'")
    if not detailsB["matches_original"]:
        print(f"    Modified reconstructed: '{detailsB["reconstructed"]}'")
    
    return detailsA, detailsB

def store_tokenization_info(item_dict: dict, textA: str, textB: str, entity: str, intervention: str):
    """Store detailed tokenization information in the item dictionary."""
    detailsA = get_tokenization_details(textA)
    detailsB = get_tokenization_details(textB)
    
    item_dict["tokenization_info"] = {
        "entity": entity,
        "intervention": intervention,
        "token_change": detailsB["token_count"] - detailsA["token_count"],
        "original": detailsA,
        "modified": detailsB
    }
    
    return item_dict

print("Enhanced tokenization logging functions loaded!")

In [ ]:
# Enhanced tokenization logging and analysis functions
def get_tokenization_details(text: str) -> dict:
    """Get detailed tokenization information for a text string."""
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    tokens_decoded = [tokenizer.decode([token_id]) for token_id in token_ids]
    reconstructed = tokenizer.decode(token_ids, skip_special_tokens=True)
    
    return {
        'text': text,
        'token_count': len(token_ids),
        'token_ids': token_ids,
        'tokens_decoded': tokens_decoded,
        'reconstructed': reconstructed,
        'matches_original': text == reconstructed
    }

def log_tokenization_comparison(textA: str, textB: str, entity: str, intervention: str):
    """Log detailed tokenization comparison between A and B texts."""
    detailsA = get_tokenization_details(textA)
    detailsB = get_tokenization_details(textB)
    
    print(f"\n--- TOKENIZATION COMPARISON: {intervention.upper()} ---")
    print(f"Entity: '{entity}'")
    print(f"Original (A): '{textA}'")
    print(f"Modified (B): '{textB}'")
    print(f"Token change: +{detailsB['token_count'] - detailsA['token_count']}")
    
    print(f"\nDetailed breakdown:")
    print(f"  Original tokens ({detailsA['token_count']}):")
    for i, token in enumerate(detailsA['tokens_decoded']):
        print(f"    [{i}] '{token}'")
    
    print(f"  Modified tokens ({detailsB['token_count']}):")
    for i, token in enumerate(detailsB['tokens_decoded']):
        print(f"    [{i}] '{token}'")
    
    print(f"\nReconstruction verification:")
    print(f"  Original matches: {detailsA['matches_original']}")
    print(f"  Modified matches: {detailsB['matches_original']}")
    if not detailsA['matches_original']:
        print(f"    Original reconstructed: '{detailsA['reconstructed']}'")
    if not detailsB['matches_original']:
        print(f"    Modified reconstructed: '{detailsB['reconstructed']}'")
    
    return detailsA, detailsB

def store_tokenization_info(item_dict: dict, textA: str, textB: str, entity: str, intervention: str):
    """Store detailed tokenization information in the item dictionary."""
    detailsA = get_tokenization_details(textA)
    detailsB = get_tokenization_details(textB)
    
    item_dict['tokenization_info'] = {
        'entity': entity,
        'intervention': intervention,
        'token_change': detailsB['token_count'] - detailsA['token_count'],
        'original': detailsA,
        'modified': detailsB
    }
    
    return item_dict

print("Enhanced tokenization logging functions loaded!")


In [9]:
# Sanity check: verify that interventions successfully converted entities from 1→2 tokens
def verify_tokenization_in_context(text: str, ent: str) -> int:
    """Get token count for entity in context, with fallback to isolated count."""
    offsets, _ = get_offsets(text)
    if offsets is None:
        return tokenized_length(ent)
    
    start = text.lower().find(ent.lower())
    if start == -1:
        return tokenized_length(ent)
    
    end = start + len(ent)
    ts = te = None
    for i, (cs, ce) in enumerate(offsets):
        if cs <= start < ce and ts is None:
            ts = i
        if cs < end <= ce:
            te = i + 1
            break
    return (te - ts) if ts is not None and te is not None else tokenized_length(ent)

def find_entity_in_text(text: str, entity: str) -> int:
    """Find the position of entity in text (case-insensitive)."""
    return text.lower().find(entity.lower())

def verify_entity_tokenization_in_context(text: str, entity: str) -> int:
    """Get token count for entity in context, with better fallback."""
    # First try with offset mapping
    offsets, _ = get_offsets(text)
    if offsets is not None:
        start = find_entity_in_text(text, entity)
        if start != -1:
            end = start + len(entity)
            ts = te = None
            for i, (cs, ce) in enumerate(offsets):
                if cs <= start < ce and ts is None:
                    ts = i
                if cs < end <= ce:
                    te = i + 1
                    break
            if ts is not None and te is not None:
                return te - ts
    
    # Fallback to isolated tokenization
    return tokenized_length(entity)

sanity_results = {}
for intervention_name, rows in per_intervention.items():
    valid_examples = 0
    invalid_examples = []
    total_entities_found = 0
    
    for s in rows:
        textA, textB, entities = s["textA"], s["textB"], s["entities"]
        dataset = s["dataset"]
        
        # Basic validation: check that textA != textB and entities were wrapped
        if textA == textB:
            invalid_examples.append({
                "id": s["id"],
                "dataset": dataset,
                "textA": textA,
                "textB": textB,
                "entities": entities,
                "issue": "No change between textA and textB"
            })
            continue
        
        # Check that textB contains wrapped entities
        wrapped_entities_found = 0
        tokenization_details = []
        
        for ent in entities:
            # Get the wrapper for this intervention
            for transform_name, pre, post in TRANSFORMS:
                if transform_name == intervention_name:
                    wrapped_ent = pre + ent + post
                    if wrapped_ent in textB:
                        wrapped_entities_found += 1
                    
                    # Get tokenization info for debugging
                    isolated_tokens_original = tokenized_length(ent)
                    isolated_tokens_wrapped = tokenized_length(wrapped_ent)
                    
                    tokenization_details.append({
                        'entity': ent,
                        'wrapped': wrapped_ent,
                        'found_in_textB': wrapped_ent in textB,
                        'original_tokens': isolated_tokens_original,
                        'wrapped_tokens': isolated_tokens_wrapped
                    })
                    break
        
        total_entities_found += wrapped_entities_found
        
        # Consider valid if we found wrapped entities and token counts increased
        wrapped_rate = wrapped_entities_found / len(entities) if entities else 0
        
        # Also check that total token count increased (indication of more tokenization)
        tokens_A = len(tokenizer.encode(textA, add_special_tokens=False))
        tokens_B = len(tokenizer.encode(textB, add_special_tokens=False))
        token_increase = tokens_B > tokens_A
        
        if wrapped_rate >= 0.8 and token_increase:  # More lenient: 80% of entities wrapped
            valid_examples += 1
        else:
            invalid_examples.append({
                "id": s["id"],
                "dataset": dataset,
                "textA": textA,
                "textB": textB,
                "entities": entities,
                "wrapped_rate": wrapped_rate,
                "tokens_A": tokens_A,
                "tokens_B": tokens_B,
                "token_increase": token_increase,
                "tokenization_details": tokenization_details[:3]  # Show first 3
            })
    
    sanity_results[intervention_name] = {
        "valid_examples": valid_examples,
        "invalid_examples": len(invalid_examples),
        "total_entities_found": total_entities_found,
        "invalid_details": invalid_examples[:2]  # Show first 2 failures
    }

def get_detailed_tokenization(text: str, entity: str) -> str:
    """Get detailed tokenization info for debugging."""
    # Try to get the entity's tokens in context
    tokens = tokenizer.encode(text, add_special_tokens=False)
    decoded_tokens = [tokenizer.decode([t]) for t in tokens]
    full_decoded = tokenizer.decode(tokens)
    
    # Find entity in text
    pos = find_entity_in_text(text, entity)
    if pos == -1:
        return f"Entity '{entity}' not found in text '{text}'"
    
    # Approximate which tokens correspond to the entity
    # This is a simplified approach for debugging
    entity_tokens = tokenizer.encode(entity, add_special_tokens=False)
    entity_decoded = [tokenizer.decode([t]) for t in entity_tokens]
    
    return f"Text: '{text}' → Tokens: {decoded_tokens} | Entity '{entity}' → {entity_decoded} ({len(entity_decoded)} tokens)"

print("Sanity check results (intervention validation):")
for intervention_name, result in sanity_results.items():
    print(f"\n{intervention_name}: {result['valid_examples']} valid, {result['invalid_examples']} invalid")
    print(f"  Total wrapped entities found: {result['total_entities_found']}")
    
    if result['invalid_details']:
        ex = result['invalid_details'][0]
        print(f"  Example failure: {ex['dataset']} ID {ex['id']}")
        print(f"    TextA: {ex['textA']}")
        print(f"    TextB: {ex['textB']}")
        print(f"    Entities: {ex['entities']}")
        
        if 'wrapped_rate' in ex:
            print(f"    Wrapped rate: {ex['wrapped_rate']:.2f}")
            print(f"    Tokens A→B: {ex['tokens_A']} → {ex['tokens_B']} (increase: {ex['token_increase']})")
            
            # Show tokenization details
            if ex['tokenization_details']:
                print(f"    Tokenization details:")
                for detail in ex['tokenization_details']:
                    print(f"      '{detail['entity']}' → '{detail['wrapped']}': {detail['original_tokens']} → {detail['wrapped_tokens']} tokens, found: {detail['found_in_textB']}")
        
        if 'issue' in ex:
            print(f"    Issue: {ex['issue']}")

# For evaluation, use all collected examples
filtered_per_intervention = per_intervention

print(f"\nUsing all collected examples for evaluation:")
for intervention_name, items in filtered_per_intervention.items():
    dataset_counts = {}
    for item in items:
        dataset = item['dataset']
        dataset_counts[dataset] = dataset_counts.get(dataset, 0) + 1
    print(f"{intervention_name}: {len(items)} total ({dataset_counts})")


Sanity check results (intervention validation):

wrap_quotes: 400 valid, 0 invalid
  Total wrapped entities found: 1306

wrap_parentheses: 400 valid, 0 invalid
  Total wrapped entities found: 1306

wrap_brackets: 400 valid, 0 invalid
  Total wrapped entities found: 1306

wrap_unicode_quotes: 400 valid, 0 invalid
  Total wrapped entities found: 1306

prefix_space: 400 valid, 0 invalid
  Total wrapped entities found: 1309

prefix_underscore: 399 valid, 1 invalid
  Total wrapped entities found: 1304
  Example failure: textvqa ID 34830
    TextA: What town are we in?
Reference OCR token: #BURNABY, BURHABY, WELCOME, TO, FROAK
    TextB: What __town are _we in?
_Reference OCR token: #BURNABY, BURHABY, WELCOME, TO, FROAK
    Entities: ['Reference', 'we', 'TO', 'town']
    Wrapped rate: 0.75
    Tokens A→B: 34 → 37 (increase: True)
    Tokenization details:
      'Reference' → '_Reference': 1 → 2 tokens, found: True
      'we' → '_we': 1 → 2 tokens, found: True
      'TO' → '_TO': 1 → 2 tokens

In [10]:
# Run VLM evaluation on all-entities A/B prompts across datasets with batch inference
from PIL import Image
import ast
import torch
from typing import List, Tuple

per_intervention_results: Dict[str, List[Dict[str, Any]]] = {name: [] for name, _, _ in TRANSFORMS}

# Batch size for inference (process multiple examples at once)
BATCH_SIZE = 32  # Adjust based on GPU memory

def process_batch(batch_items: List[Dict[str, Any]]) -> List[Tuple[str, str]]:
    """Process a batch of items and return (outA, outB) pairs."""
    images = []
    promptsA = []
    promptsB = []
    
    # Prepare batch data
    for s in batch_items:
        img = Image.open(s["img_path"]).convert("RGB")
        textA = s["textA"]
        textB = s["textB"]
        dataset = s["dataset"]
        
        # Get appropriate prompts for this dataset
        prompt_fn = prompt_fns[dataset]
        
        if dataset == 'tallyqa':
            promptA = prompt_fn(textA, choices=[str(i) for i in range(16)])
            promptB = prompt_fn(textB, choices=[str(i) for i in range(16)])
        else:
            promptA = prompt_fn(textA)
            promptB = prompt_fn(textB)
        
        images.append(img)
        promptsA.append(promptA)
        promptsB.append(promptB)
    
    # Process images in batch
    if hasattr(image_processor, "__call__"):
        # Process all images at once
        pixel_values_list = []
        for img in images:
            pixel_values_single = image_processor(img, return_tensors="pt")["pixel_values"][0]
            pixel_values_single = pixel_values_single.to(vlm.distributed_state.device)
            pixel_values_list.append(pixel_values_single)
        
        # Stack all images: [batch_size, channels, height, width]
        pixel_values_batch = torch.stack(pixel_values_list, dim=0)
        
        # For A/B comparison, we need to duplicate each image and interleave prompts
        # Final shape: [batch_size * 2, channels, height, width] 
        pixel_values_doubled = torch.repeat_interleave(pixel_values_batch, 2, dim=0)
        
        # Interleave prompts: [promptA1, promptB1, promptA2, promptB2, ...]
        prompts_interleaved = []
        for pA, pB in zip(promptsA, promptsB):
            prompts_interleaved.extend([pA, pB])
        
        # Generate answers for the entire batch
        outs = vlm.generate_answer(pixel_values_doubled, prompts_interleaved)
        
        # Separate A and B outputs: outs[0::2] are A, outs[1::2] are B
        outsA = outs[0::2]  # [0, 2, 4, ...]
        outsB = outs[1::2]  # [1, 3, 5, ...]
        
        return list(zip(outsA, outsB))
    else:
        raise RuntimeError("Unexpected image_processor type")

print("Running VLM evaluation across all datasets with batch inference...")
print(f"Using batch size: {BATCH_SIZE}")

for intervention_name, items in filtered_per_intervention.items():
    print(f"\nProcessing {intervention_name}: {len(items)} examples")
    
    # Process items in batches
    for batch_start in range(0, len(items), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, len(items))
        batch_items = items[batch_start:batch_end]
        
        print(f"  Batch {batch_start//BATCH_SIZE + 1}/{(len(items) + BATCH_SIZE - 1)//BATCH_SIZE}: "
              f"examples {batch_start}-{batch_end-1}")
        
        try:
            # Process the entire batch at once
            batch_outputs = process_batch(batch_items)
            
            # Store results
            for i, (s, (outA, outB)) in enumerate(zip(batch_items, batch_outputs)):
                per_intervention_results[intervention_name].append({
                    "id": s["id"],
                    "dataset": s["dataset"],
                    "gt": s["answer"],
                    "entities": s["entities"],
                    "entity_count": s["entity_count"],
                    "textA": s["textA"],
                    "textB": s["textB"],
                    "outA": outA,
                    "outB": outB,
                })
                
        except Exception as e:
            print(f"  Error processing batch {batch_start//BATCH_SIZE + 1}: {e}")
            print("  Falling back to individual processing for this batch...")
            
            # Fallback: process individually
            for i, s in enumerate(batch_items):
                try:
                    img = Image.open(s["img_path"]).convert("RGB")
                    textA = s["textA"]
                    textB = s["textB"]
                    dataset = s["dataset"]
                    
                    # Get appropriate prompts for this dataset
                    prompt_fn = prompt_fns[dataset]
                    
                    if dataset == 'tallyqa':
                        promptA = prompt_fn(textA, choices=[str(i) for i in range(16)])
                        promptB = prompt_fn(textB, choices=[str(i) for i in range(16)])
                    else:
                        promptA = prompt_fn(textA)
                        promptB = prompt_fn(textB)

                    # Process image
                    pixel_values_single = image_processor(img, return_tensors="pt")["pixel_values"][0]
                    pixel_values_single = pixel_values_single.to(vlm.distributed_state.device)
                    pixel_values = torch.stack([pixel_values_single, pixel_values_single], dim=0)

                    # Generate answers
                    outs = vlm.generate_answer(pixel_values, [promptA, promptB])
                    outA, outB = outs[0], outs[1]

                    per_intervention_results[intervention_name].append({
                        "id": s["id"],
                        "dataset": dataset,
                        "gt": s["answer"],
                        "entities": s["entities"],
                        "entity_count": s["entity_count"],
                        "textA": textA,
                        "textB": textB,
                        "outA": outA,
                        "outB": outB,
                    })
                    
                except Exception as e2:
                    print(f"    Error processing individual example {batch_start + i}: {e2}")
                    continue

print("\nCompleted VLM evaluation. Results per intervention:")
for intervention_name, results in per_intervention_results.items():
    dataset_counts = {}
    for result in results:
        dataset = result['dataset']
        dataset_counts[dataset] = dataset_counts.get(dataset, 0) + 1
    print(f"{intervention_name}: {len(results)} total ({dataset_counts})")


Running VLM evaluation across all datasets with batch inference...
Using batch size: 32

Processing wrap_quotes: 400 examples
  Batch 1/13: examples 0-31


/localdisk/ssrivas9/miniconda3/envs/eval/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.2` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


  Batch 2/13: examples 32-63
  Batch 3/13: examples 64-95
  Batch 4/13: examples 96-127
  Batch 5/13: examples 128-159
  Batch 6/13: examples 160-191
  Batch 7/13: examples 192-223
  Batch 8/13: examples 224-255
  Batch 9/13: examples 256-287
  Batch 10/13: examples 288-319
  Batch 11/13: examples 320-351
  Batch 12/13: examples 352-383
  Batch 13/13: examples 384-399

Processing wrap_parentheses: 400 examples
  Batch 1/13: examples 0-31
  Batch 2/13: examples 32-63
  Batch 3/13: examples 64-95
  Batch 4/13: examples 96-127
  Batch 5/13: examples 128-159
  Batch 6/13: examples 160-191
  Batch 7/13: examples 192-223
  Batch 8/13: examples 224-255
  Batch 9/13: examples 256-287
  Batch 10/13: examples 288-319
  Batch 11/13: examples 320-351
  Batch 12/13: examples 352-383
  Batch 13/13: examples 384-399

Processing wrap_brackets: 400 examples
  Batch 1/13: examples 0-31
  Batch 2/13: examples 32-63
  Batch 3/13: examples 64-95
  Batch 4/13: examples 96-127
  Batch 5/13: examples 128-159


In [11]:
# Evaluate correctness metrics per intervention and dataset

def normalize_ans(s: str) -> str:
    """Normalize answer string."""
    return s.strip().lower()

def compute_iou(pred_bbox: List[float], gt_bbox: List[float]) -> float:
    """Compute IoU between two bboxes in xyxy format."""
    int_x1, int_y1 = max(pred_bbox[0], gt_bbox[0]), max(pred_bbox[1], gt_bbox[1])
    int_x2, int_y2 = min(pred_bbox[2], gt_bbox[2]), min(pred_bbox[3], gt_bbox[3])

    # Compute Box Areas
    pred_area = (pred_bbox[2] - pred_bbox[0]) * (pred_bbox[3] - pred_bbox[1])
    gt_area = (gt_bbox[2] - gt_bbox[0]) * (gt_bbox[3] - gt_bbox[1])

    # Compute Intersection Area
    intersection_area = max(0, int_x2 - int_x1) * max(0, int_y2 - int_y1)

    # Compute Union Area
    union_area = pred_area + gt_area - intersection_area

    # Return IOU
    return intersection_area / union_area if union_area > 0 else 0.0

def parse_bbox(bbox_str: str) -> Optional[List[float]]:
    """Parse bbox string to list of floats."""
    try:
        bbox = ast.literal_eval(bbox_str)
        if isinstance(bbox, list) and len(bbox) == 4:
            return [float(x) for x in bbox]
    except:
        pass
    return None

def evaluate_dataset_specific(dataset: str, gt: str, outA: str, outB: str) -> Tuple[bool, bool]:
    """Evaluate correctness for specific dataset type."""
    if dataset == 'tallyqa':
        # Enhanced TallyQA evaluation with comprehensive logging
        import re
        import logging
        
        # Initialize logging for TallyQA if not already done
        if not hasattr(evaluate_dataset_specific, 'tallyqa_logger'):
            evaluate_dataset_specific.tallyqa_logger = logging.getLogger('TallyQA_Evaluation')
            evaluate_dataset_specific.tallyqa_logger.setLevel(logging.INFO)
            if not evaluate_dataset_specific.tallyqa_logger.handlers:
                handler = logging.StreamHandler()
                formatter = logging.Formatter('%(name)s - %(levelname)s - %(message)s')
                handler.setFormatter(formatter)
                evaluate_dataset_specific.tallyqa_logger.addHandler(handler)
        
        logger = evaluate_dataset_specific.tallyqa_logger
        
        # Enhanced ground truth parsing with detailed logging
        gt_num = None
        try:
            if isinstance(gt, str):
                gt_num = int(gt)
                logger.debug(f"Ground truth parsed as string: '{gt}' -> {gt_num}")
            elif isinstance(gt, (int, float)):
                gt_num = int(gt)
                logger.debug(f"Ground truth parsed as number: {gt} ({type(gt).__name__}) -> {gt_num}")
            else:
                logger.error(f"Ground truth has unexpected type: {gt} (type: {type(gt).__name__})")
                return False, False
        except (ValueError, TypeError) as e:
            logger.error(f"Failed to parse ground truth: {gt} (type: {type(gt).__name__}) - Error: {e}")
            return False, False
        
        # Initialize counters for detailed tracking
        if not hasattr(evaluate_dataset_specific, 'tallyqa_stats'):
            evaluate_dataset_specific.tallyqa_stats = {
                'total_processed': 0,
                'successful_extractions_A': 0,
                'successful_extractions_B': 0,
                'failed_extractions_A': 0,
                'failed_extractions_B': 0,
                'correct_predictions_A': 0,
                'correct_predictions_B': 0,
                'extraction_methods': {
                    'direct_int': 0,
                    'regex_pattern': 0,
                    'word_number': 0,
                    'negation': 0,
                    'failed': 0
                }
            }
        
        stats = evaluate_dataset_specific.tallyqa_stats
        stats['total_processed'] += 1
        
        def extract_number_with_logging(text: str, prediction_type: str) -> tuple:
            """Extract number with detailed logging of the extraction process."""
            original_text = text
            text = text.strip().lower()
            extraction_method = 'unknown'
            
            logger.debug(f"Extracting number from {prediction_type}: '{original_text}' -> normalized: '{text}'")
            
            # Method 1: Direct integer parsing
            try:
                result = int(text)
                extraction_method = 'direct_int'
                stats['extraction_methods']['direct_int'] += 1
                logger.debug(f"{prediction_type}: Direct int conversion successful: {result}")
                return result, extraction_method, None
            except ValueError:
                logger.debug(f"{prediction_type}: Direct int conversion failed")
                pass
            
            # Method 2: Regex patterns for numbers with context
            patterns = [
                (r'(\d+)\.', 'number with period'),
                (r'(\d+)\s', 'number with space'),
                (r'(\d+)$', 'number at end'),
                (r'(\d+)', 'any number')
            ]
            
            for pattern, description in patterns:
                match = re.search(pattern, text)
                if match:
                    try:
                        result = int(match.group(1))
                        extraction_method = 'regex_pattern'
                        stats['extraction_methods']['regex_pattern'] += 1
                        logger.debug(f"{prediction_type}: Regex extraction successful ({description}): {result}")
                        return result, extraction_method, None
                    except ValueError:
                        logger.debug(f"{prediction_type}: Regex match failed to convert: {match.group(1)}")
                        continue
            
            # Method 3: Word numbers
            word_to_num = {
                'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
                'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
                'eleven': 11, 'twelve': 12, 'thirteen': 13, 'fourteen': 14, 'fifteen': 15
            }
            
            for word, num in word_to_num.items():
                if word in text:
                    extraction_method = 'word_number'
                    stats['extraction_methods']['word_number'] += 1
                    logger.debug(f"{prediction_type}: Word number extraction successful: '{word}' -> {num}")
                    return num, extraction_method, None
            
            # Method 4: Negation patterns (indicating zero)
            negative_patterns = ['no ', 'none', 'not any', 'cannot see', 'no visible', 'are not', 'zero', 'nothing']
            for pattern in negative_patterns:
                if pattern in text:
                    extraction_method = 'negation'
                    stats['extraction_methods']['negation'] += 1
                    logger.debug(f"{prediction_type}: Negation pattern detected: '{pattern}' -> 0")
                    return 0, extraction_method, None
            
            # Method 5: Look for choice letters (A, B, C, etc.) and convert to numbers
            choice_match = re.search(r'\b([a-p])\b', text)
            if choice_match:
                letter = choice_match.group(1).upper()
                if 'A' <= letter <= 'P':
                    result = ord(letter) - ord('A')  # A=0, B=1, etc.
                    extraction_method = 'choice_letter'
                    logger.debug(f"{prediction_type}: Choice letter extraction: '{letter}' -> {result}")
                    return result, extraction_method, None
            
            # All methods failed
            extraction_method = 'failed'
            stats['extraction_methods']['failed'] += 1
            error_msg = f"Could not extract number from: '{original_text}' (normalized: '{text}')"
            logger.warning(f"{prediction_type}: {error_msg}")
            return None, extraction_method, ValueError(error_msg)
        
        # Extract numbers for both predictions with detailed logging
        predA, methodA, errorA = extract_number_with_logging(outA, "Prediction_A")
        predB, methodB, errorB = extract_number_with_logging(outB, "Prediction_B")
        
        # Evaluate correctness with logging
        correctA = False
        correctB = False
        
        if predA is not None:
            correctA = (predA == gt_num)
            stats['successful_extractions_A'] += 1
            if correctA:
                stats['correct_predictions_A'] += 1
            logger.debug(f"Prediction A: {predA} vs GT: {gt_num} -> {'CORRECT' if correctA else 'INCORRECT'} (method: {methodA})")
        else:
            stats['failed_extractions_A'] += 1
            logger.warning(f"Prediction A extraction failed: '{outA}' -> Error: {errorA}")
        
        if predB is not None:
            correctB = (predB == gt_num)
            stats['successful_extractions_B'] += 1
            if correctB:
                stats['correct_predictions_B'] += 1
            logger.debug(f"Prediction B: {predB} vs GT: {gt_num} -> {'CORRECT' if correctB else 'INCORRECT'} (method: {methodB})")
        else:
            stats['failed_extractions_B'] += 1
            logger.warning(f"Prediction B extraction failed: '{outB}' -> Error: {errorB}")
        
        # Log summary statistics every 50 examples
        if stats['total_processed'] % 50 == 0:
            total = stats['total_processed']
            success_rate_A = (stats['successful_extractions_A'] / total) * 100
            success_rate_B = (stats['successful_extractions_B'] / total) * 100
            accuracy_A = (stats['correct_predictions_A'] / max(1, stats['successful_extractions_A'])) * 100
            accuracy_B = (stats['correct_predictions_B'] / max(1, stats['successful_extractions_B'])) * 100
            
            logger.info(f"TallyQA Progress Summary (after {total} examples):")
            logger.info(f"  Extraction Success: A={success_rate_A:.1f}%, B={success_rate_B:.1f}%")
            logger.info(f"  Accuracy (of successful): A={accuracy_A:.1f}%, B={accuracy_B:.1f}%")
            logger.info(f"  Extraction methods: {dict(stats['extraction_methods'])}")
        
        # Log problematic cases for the first few failures
        if not correctA and predA is not None and stats['total_processed'] <= 10:
            logger.info(f"Early incorrect example A: GT={gt_num}, Pred={predA}, Text='{outA}'")
        if not correctB and predB is not None and stats['total_processed'] <= 10:
            logger.info(f"Early incorrect example B: GT={gt_num}, Pred={predB}, Text='{outB}'")
        
        return correctA, correctB
        
    elif dataset == 'refcoco':
        # IoU > 0.5 for bounding boxes
        gt_bbox = parse_bbox(gt)
        predA_bbox = parse_bbox(outA)
        predB_bbox = parse_bbox(outB)
        
        correctA = False
        correctB = False
        
        if gt_bbox and predA_bbox:
            iou_a = compute_iou(predA_bbox, gt_bbox)
            correctA = (iou_a >= 0.5)
            
        if gt_bbox and predB_bbox:
            iou_b = compute_iou(predB_bbox, gt_bbox)
            correctB = (iou_b >= 0.5)
            
        return correctA, correctB
        
    elif dataset == 'textvqa':
        # Multiple acceptable answers (TextVQA style)
        # gt is already a list of answers from our FixedTextVQAIndexDataset
        if isinstance(gt, list):
            gt_answers = [normalize_ans(ans.strip()) for ans in gt]
        else:
            # Fallback: if gt is a string, split by ' or '
            gt_answers = [normalize_ans(ans.strip()) for ans in gt.split(' or ')]
        
        predA_norm = normalize_ans(outA)
        predB_norm = normalize_ans(outB)
        
        correctA = any(predA_norm == gt_ans for gt_ans in gt_answers)
        correctB = any(predB_norm == gt_ans for gt_ans in gt_answers)
        
        return correctA, correctB
    
    elif dataset == 'vqav2':
        # VQAv2 uses exact string matching with normalization
        predA_norm = normalize_ans(outA)
        predB_norm = normalize_ans(outB)
        gt_norm = normalize_ans(str(gt))
        
        correctA = predA_norm == gt_norm
        correctB = predB_norm == gt_norm
        
        return correctA, correctB
    
    else:
        # Fallback: exact match
        correctA = normalize_ans(outA) == normalize_ans(gt)
        correctB = normalize_ans(outB) == normalize_ans(gt)
        return correctA, correctB

# Calculate metrics per intervention and dataset
metrics = {}

for intervention_name, rows in per_intervention_results.items():
    # Group by dataset
    dataset_groups = {}
    for r in rows:
        dataset = r['dataset']
        if dataset not in dataset_groups:
            dataset_groups[dataset] = []
        dataset_groups[dataset].append(r)
    
    metrics[intervention_name] = {}
    
    # Calculate metrics for each dataset
    for dataset, dataset_rows in dataset_groups.items():
        accA = accB = 0
        
        for r in dataset_rows:
            correctA, correctB = evaluate_dataset_specific(dataset, r["gt"], r["outA"], r["outB"])
            accA += correctA
            accB += correctB
        
        n = len(dataset_rows)
        if n > 0:
            acc_a = accA / n
            acc_b = accB / n
            delta = acc_a - acc_b
            metrics[intervention_name][dataset] = {
                "n": n, 
                "accA": acc_a, 
                "accB": acc_b, 
                "delta_A_minus_B": delta
            }
        else:
            metrics[intervention_name][dataset] = {"n": 0, "accA": 0.0, "accB": 0.0, "delta_A_minus_B": 0.0}
    
    # Calculate overall metrics across all datasets
    total_accA = total_accB = total_n = 0
    for dataset_metric in metrics[intervention_name].values():
        total_accA += dataset_metric["accA"] * dataset_metric["n"]
        total_accB += dataset_metric["accB"] * dataset_metric["n"]
        total_n += dataset_metric["n"]
    
    if total_n > 0:
        overall_acc_a = total_accA / total_n
        overall_acc_b = total_accB / total_n
        metrics[intervention_name]["overall"] = {
            "n": total_n,
            "accA": overall_acc_a,
            "accB": overall_acc_b,
            "delta_A_minus_B": overall_acc_a - overall_acc_b
        }

print("Performance metrics per intervention and dataset:")
print("(A = ALL entities as 1 token each, B = ALL entities as 2 tokens each)")
print()

for intervention_name, dataset_metrics in metrics.items():
    print(f"{intervention_name}:")
    
    for dataset, metric in dataset_metrics.items():
        if dataset == "overall":
            print(f"  {dataset.upper()}: n={metric['n']}, accA={metric['accA']:.3f}, accB={metric['accB']:.3f}, delta={metric['delta_A_minus_B']:.3f}")
        else:
            print(f"    {dataset}: n={metric['n']}, accA={metric['accA']:.3f}, accB={metric['accB']:.3f}, delta={metric['delta_A_minus_B']:.3f}")
        
        if metric['delta_A_minus_B'] > 0.02:
            print(f"      → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)")
        elif metric['delta_A_minus_B'] < -0.02:
            print(f"      → B (all entities 2-tokens) performs BETTER than A (all entities 1-token)")
    print()

# Summary
print("Summary:")
total_examples = sum(metrics[name]["overall"]["n"] for name in metrics.keys() if "overall" in metrics[name])
print(f"Total examples across all interventions and datasets: {total_examples}")

significant_deltas = []
for intervention_name, dataset_metrics in metrics.items():
    for dataset, metric in dataset_metrics.items():
        if abs(metric['delta_A_minus_B']) > 0.05:
            significant_deltas.append(f"{intervention_name}-{dataset}")

print(f"Intervention-dataset pairs with >5% performance difference: {significant_deltas}")


TallyQA_Evaluation - INFO - Early incorrect example A: GT=6, Pred=0, Text='A'


08/12 [09:09:48] INFO     | >> Early incorrect example A: GT=6, Pred=0, Text='A'                  ]8;id=772246;file:///tmp/ipykernel_3819400/3258839850.py\3258839850.py]8;;\:]8;id=107473;file:///tmp/ipykernel_3819400/3258839850.py#215\215]8;;\

TallyQA_Evaluation - INFO - Early incorrect example B: GT=6, Pred=0, Text='A'


                 INFO     | >> Early incorrect example B: GT=6, Pred=0, Text='A'                  ]8;id=619176;file:///tmp/ipykernel_3819400/3258839850.py\3258839850.py]8;;\:]8;id=442417;file:///tmp/ipykernel_3819400/3258839850.py#217\217]8;;\

TallyQA_Evaluation - INFO - Early incorrect example B: GT=0, Pred=2, Text='C'


...

TallyQA_Evaluation - INFO -   Accuracy (of successful): A=64.0%, B=62.8%


                 INFO     | >>   Accuracy (of successful): A=64.0%, B=62.8%                       ]8;id=355784;file:///tmp/ipykernel_3819400/3258839850.py\3258839850.py]8;;\:]8;id=839482;file:///tmp/ipykernel_3819400/3258839850.py#210\210]8;;\

TallyQA_Evaluation - INFO -   Extraction methods: {'direct_int': 0, 'regex_pattern': 0, 'word_number': 0, 'negation': 0, 'failed': 0}


                 INFO     | >>   Extraction methods: {'direct_int': 0, 'regex_pattern': 0,        ]8;id=199448;file:///tmp/ipykernel_3819400/3258839850.py\3258839850.py]8;;\:]8;id=562336;file:///tmp/ipykernel_3819400/3258839850.py#211\211]8;;\
                          'word_number': 0, 'negation': 0, 'failed': 0}                                            

Performance metrics per intervention and dataset:
(A = ALL entities as 1 token each, B = ALL entities as 2 tokens each)

wrap_quotes:
    tallyqa: n=100, accA=0.640, accB=0.620, delta=0.020
      → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)
    refcoco: n=100, accA=0.520, accB=0.480, delta=0.040
      → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)
    textvqa: n=100, accA=0.550, accB=0.530, delta=0.020
      → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)
    vqav2: n=100, accA=0.610, accB=0.570, delta=0.040
      → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)
  OVERALL: n=400, accA=0.580, accB=0.550, delta=0.030
      → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)

wrap_parentheses:
    tallyqa: n=100, accA=0.640, accB=0.610, delta=0.030
      → A (all entities 1-token) performs BETTER than B (all entities 2-tokens)
    refcoco: n=100, accA=0.520, acc

In [12]:
# Create comprehensive output files for review
import json
import os
from pathlib import Path

# Create interventions directory structure
INTERVENTIONS_DIR = Path("/localdisk/ssrivas9/vlm-evaluation/interventions")
INTERVENTIONS_DIR.mkdir(exist_ok=True)

print(f"Creating output files in {INTERVENTIONS_DIR}")

# For each intervention, create a subfolder with dataset-specific files
for intervention_name, results in per_intervention_results.items():
    intervention_dir = INTERVENTIONS_DIR / intervention_name
    intervention_dir.mkdir(exist_ok=True)
    
    # Group results by dataset
    dataset_results = {}
    for result in results:
        dataset = result['dataset']
        if dataset not in dataset_results:
            dataset_results[dataset] = []
        dataset_results[dataset].append(result)
    
    # Save each dataset's results
    for dataset, dataset_items in dataset_results.items():
        # Create detailed JSON file
        json_file = intervention_dir / f"{dataset}.json"
        
        # Prepare data for JSON serialization
        json_data = {
            "intervention": intervention_name,
            "dataset": dataset,
            "total_examples": len(dataset_items),
            "examples": []
        }
        
        for item in dataset_items:
            # Calculate correctness for this item
            correctA, correctB = evaluate_dataset_specific(dataset, item["gt"], item["outA"], item["outB"])
            
            json_data["examples"].append({
                "id": item["id"],
                "entities": item["entities"],
                "entity_count": item["entity_count"],
                "question_A": item["textA"],
                "question_B": item["textB"],
                "ground_truth": item["gt"],
                "prediction_A": item["outA"],
                "prediction_B": item["outB"],
                "correct_A": correctA,
                "correct_B": correctB,
                "image_path": str(item.get("img_path", "N/A"))
                "tokenization": item.get("tokenization_info", {}),
            })
        
        # Save JSON file
        with open(json_file, 'w') as f:
            json.dump(json_data, f, indent=2)
        
        # Create human-readable text file
        txt_file = intervention_dir / f"{dataset}.txt"
        with open(txt_file, 'w') as f:
            f.write(f"Intervention: {intervention_name}\n")
            f.write(f"Dataset: {dataset}\n")
            f.write(f"Total examples: {len(dataset_items)}\n")
            f.write("=" * 80 + "\n\n")
            
            for i, item in enumerate(dataset_items, 1):
                correctA, correctB = evaluate_dataset_specific(dataset, item["gt"], item["outA"], item["outB"])
                
                f.write(f"Example {i} (ID: {item['id']}):\n")
                f.write(f"  Entities modified ({item['entity_count']}): {', '.join(item['entities'])}\n")
                f.write(f"  Question A: {item['textA']}\n")
                f.write(f"  Question B: {item['textB']}\n")
                f.write(f"  Ground Truth: {item['gt']}\n")
                f.write(f"  Prediction A: {item['outA']} ({'✓' if correctA else '✗'})\n")
                f.write(f"  Prediction B: {item['outB']} ({'✓' if correctB else '✗'})\n")
                f.write(f"  Image: {item.get('img_path', 'N/A')}\n")
                \n
                # Add tokenization breakdown\n
                tokenization_info = item.get("tokenization_info", {})\n
                if tokenization_info:\n
                    original = tokenization_info.get("original", {})\n
                    modified = tokenization_info.get("modified", {})\n
                    f.write(f"  Tokenization Analysis:\\n")\n
                    f.write(f"    Token change: +{tokenization_info.get(\"token_change\", 0)}\\n")\n
                    f.write(f"    Original tokens ({original.get(\"token_count\", 0)}): {original.get(\"tokens_decoded\", [])}\\n")\n
                    f.write(f"    Modified tokens ({modified.get(\"token_count\", 0)}): {modified.get(\"tokens_decoded\", [])}\\n")\n
                    entity_name = tokenization_info.get(\"entity\", \"unknown\")\n
                    f.write(f"    Entity {entity_name} tokenization change\\n")\n
                \n
                f.write("-" * 40 + "\n\n")
    
    print(f"  {intervention_name}: {len(dataset_results)} datasets saved")

# Create summary file
summary_file = INTERVENTIONS_DIR / "summary.json"
summary_data = {
    "total_interventions": len(per_intervention_results),
    "total_examples": sum(len(results) for results in per_intervention_results.values()),
    "interventions": {},
    "metrics": {}
}

# Add intervention summaries
for intervention_name, results in per_intervention_results.items():
    dataset_breakdown = {}
    for result in results:
        dataset = result['dataset']
        dataset_breakdown[dataset] = dataset_breakdown.get(dataset, 0) + 1
    
    summary_data["interventions"][intervention_name] = {
        "total_examples": len(results),
        "datasets": dataset_breakdown
    }

# Add metrics
for intervention_name, dataset_metrics in metrics.items():
    summary_data["metrics"][intervention_name] = {}
    for dataset, metric in dataset_metrics.items():
        summary_data["metrics"][intervention_name][dataset] = {
            "n": metric["n"],
            "accuracy_A": round(metric["accA"], 3),
            "accuracy_B": round(metric["accB"], 3),
            "delta_A_minus_B": round(metric["delta_A_minus_B"], 3)
        }

with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"\nOutput files created:")
print(f"  - Summary: {summary_file}")
for intervention_name in per_intervention_results.keys():
    intervention_dir = INTERVENTIONS_DIR / intervention_name
    files = list(intervention_dir.glob("*"))
    print(f"  - {intervention_name}: {len(files)} files in {intervention_dir}")

print(f"\nTotal files created: {len(list(INTERVENTIONS_DIR.rglob('*')))} files")
print(f"Directory structure: /interventions/[intervention_name]/[dataset].[json|txt]")


TallyQA_Evaluation - INFO - TallyQA Progress Summary (after 650 examples):


Creating output files in /localdisk/ssrivas9/vlm-evaluation/interventions


                 INFO     | >> TallyQA Progress Summary (after 650 examples):                     ]8;id=292075;file:///tmp/ipykernel_3819400/3258839850.py\3258839850.py]8;;\:]8;id=485100;file:///tmp/ipykernel_3819400/3258839850.py#208\208]8;;\

TallyQA_Evaluation - INFO -   Extraction Success: A=100.0%, B=100.0%


                 INFO     | >>   Extraction Success: A=100.0%, B=100.0%                           ]8;id=464656;file:///tmp/ipykernel_3819400/3258839850.py\3258839850.py]8;;\:]8;id=847272;file:///tmp/ipykernel_3819400/3258839850.py#209\209]8;;\

...

TallyQA_Evaluation - INFO -   Accuracy (of successful): A=64.0%, B=62.8%


                 INFO     | >>   Accuracy (of successful): A=64.0%, B=62.8%                       ]8;id=876325;file:///tmp/ipykernel_3819400/3258839850.py\3258839850.py]8;;\:]8;id=439375;file:///tmp/ipykernel_3819400/3258839850.py#210\210]8;;\

TallyQA_Evaluation - INFO -   Extraction methods: {'direct_int': 0, 'regex_pattern': 0, 'word_number': 0, 'negation': 0, 'failed': 0}


                 INFO     | >>   Extraction methods: {'direct_int': 0, 'regex_pattern': 0,        ]8;id=838722;file:///tmp/ipykernel_3819400/3258839850.py\3258839850.py]8;;\:]8;id=632556;file:///tmp/ipykernel_3819400/3258839850.py#211\211]8;;\
                          'word_number': 0, 'negation': 0, 'failed': 0}                                            

  prefix_underscore: 4 datasets saved

Output files created:
  - Summary: /localdisk/ssrivas9/vlm-evaluation/interventions/summary.json
  - wrap_quotes: 8 files in /localdisk/ssrivas9/vlm-evaluation/interventions/wrap_quotes
  - wrap_parentheses: 8 files in /localdisk/ssrivas9/vlm-evaluation/interventions/wrap_parentheses
  - wrap_brackets: 8 files in /localdisk/ssrivas9/vlm-evaluation/interventions/wrap_brackets
  - wrap_unicode_quotes: 8 files in /localdisk/ssrivas9/vlm-evaluation/interventions/wrap_unicode_quotes
  - prefix_space: 8 files in /localdisk/ssrivas9/vlm-evaluation/interventions/prefix_space
  - prefix_underscore: 8 files in /localdisk/ssrivas9/vlm-evaluation/interventions/prefix_underscore

Total files created: 55 files
Directory structure: /interventions/[intervention_name]/[dataset].[json|txt]


In [13]:
# Display comprehensive TallyQA evaluation diagnostics
print("="*80)
print("TALLYQA EVALUATION DIAGNOSTICS")
print("="*80)

# Check if we have TallyQA statistics
if hasattr(evaluate_dataset_specific, 'tallyqa_stats'):
    stats = evaluate_dataset_specific.tallyqa_stats
    total = stats['total_processed']
    
    if total > 0:
        print(f"\nTotal TallyQA examples processed: {total}")
        print(f"\nExtraction Success Rates:")
        print(f"  Prediction A: {stats['successful_extractions_A']}/{total} ({stats['successful_extractions_A']/total*100:.1f}%)")
        print(f"  Prediction B: {stats['successful_extractions_B']}/{total} ({stats['successful_extractions_B']/total*100:.1f}%)")
        
        print(f"\nAccuracy Rates (of successful extractions):")
        if stats['successful_extractions_A'] > 0:
            acc_A = stats['correct_predictions_A'] / stats['successful_extractions_A'] * 100
            print(f"  Prediction A: {stats['correct_predictions_A']}/{stats['successful_extractions_A']} ({acc_A:.1f}%)")
        else:
            print(f"  Prediction A: 0/0 (N/A - no successful extractions)")
            
        if stats['successful_extractions_B'] > 0:
            acc_B = stats['correct_predictions_B'] / stats['successful_extractions_B'] * 100
            print(f"  Prediction B: {stats['correct_predictions_B']}/{stats['successful_extractions_B']} ({acc_B:.1f}%)")
        else:
            print(f"  Prediction B: 0/0 (N/A - no successful extractions)")
        
        print(f"\nExtraction Methods Used:")
        total_methods = sum(stats["extraction_methods"].values())
        if total_methods > 0:
            for method, count in stats["extraction_methods"].items():
                percentage = count / total_methods * 100
                print(f"  {method}: {count} ({percentage:.1f}%)")
        else:
            print("  No extraction methods used yet (no examples processed)")
            for method, count in stats["extraction_methods"].items():
                print(f"  {method}: {count} (0.0%)")
        print(f"\nDetailed Breakdown:")
        print(f"  Failed extractions A: {stats['failed_extractions_A']}")
        print(f"  Failed extractions B: {stats['failed_extractions_B']}")
        print(f"  Incorrect predictions A: {stats['successful_extractions_A'] - stats['correct_predictions_A']}")
        print(f"  Incorrect predictions B: {stats['successful_extractions_B'] - stats['correct_predictions_B']}")
        
        # Calculate overall TallyQA accuracy for comparison with other datasets
        total_attempts_A = stats['successful_extractions_A'] + stats['failed_extractions_A']
        total_attempts_B = stats['successful_extractions_B'] + stats['failed_extractions_B']
        overall_acc_A = stats['correct_predictions_A'] / total_attempts_A * 100 if total_attempts_A > 0 else 0
        overall_acc_B = stats['correct_predictions_B'] / total_attempts_B * 100 if total_attempts_B > 0 else 0
        
        print(f"\nOverall TallyQA Accuracy (including failed extractions as incorrect):")
        print(f"  Prediction A: {stats['correct_predictions_A']}/{total_attempts_A} ({overall_acc_A:.1f}%)")
        print(f"  Prediction B: {stats['correct_predictions_B']}/{total_attempts_B} ({overall_acc_B:.1f}%)")
        
        # Recommendations based on results
        print(f"\nDiagnostic Recommendations:")
        if stats['extraction_methods']['failed'] > total * 0.1:
            print(f"  ⚠️  High failure rate ({stats['extraction_methods']['failed']/total*100:.1f}%) suggests VLM responses may not contain extractable numbers")
            print(f"      Consider examining actual VLM outputs to improve extraction logic")
        
        if stats['extraction_methods']['direct_int'] < total * 0.5:
            print(f"  ℹ️  Low direct integer extraction rate ({stats['extraction_methods']['direct_int']/total*100:.1f}%) suggests VLM generates verbose responses")
            print(f"      This is normal for conversational models like LLaVA")
        
        if overall_acc_A < 10 and overall_acc_B < 10:
            print(f"  🔍 Very low accuracy suggests potential issues:")
            print(f"      - Check if ground truth format matches expectations")
            print(f"      - Verify VLM is generating reasonable counting responses")
            print(f"      - Consider if TallyQA prompts are appropriate for the model")
        
    else:
        print("No TallyQA examples were processed yet.")
else:
    print("TallyQA evaluation statistics not available. Run the evaluation first.")

print("="*80)


TALLYQA EVALUATION DIAGNOSTICS

Total TallyQA examples processed: 1800

Extraction Success Rates:
  Prediction A: 1800/1800 (100.0%)
  Prediction B: 1800/1800 (100.0%)

Accuracy Rates (of successful extractions):
  Prediction A: 1152/1800 (64.0%)
  Prediction B: 1131/1800 (62.8%)

Extraction Methods Used:


ZeroDivisionError: division by zero

In [ ]:
### Why can a word tokenize to one token or two in context?

- **Byte Pair Encoding (BPE)** and similar subword algorithms segment text into frequent chunks from a learned vocabulary.
- The same surface word can map to different token sequences depending on:
  - **Surrounding characters** (spaces, punctuation, quotes) that change pre-tokenization normalization and word boundaries.
  - **Casing or accents** that break merges.
  - **Whitespace handling**: some tokenizers learn merges that apply only when preceded by a space (e.g., Llama-style leading space rules).
- **Practically**: if a subword merge exists for "word" but only when preceded by a space, placing it within quotes, parentheses, or after special characters can disable that merge and split it into two tokens.

### This notebook's approach (ALL ENTITIES + MULTI-DATASET VERSION):

We enforce A vs B by applying minimal interventions around ALL eligible entities in each question/expression across multiple datasets:
- **Quotes**: `cat` and `dog` → `"cat"` and `"dog"` 
- **Parentheses**: `cat` and `dog` → `(cat)` and `(dog)`
- **Brackets**: `cat` and `dog` → `[cat]` and `[dog]`

For each intervention type, we identify ALL noun/adjective/adverb entities that tokenize as 1 token in the original text, then apply the intervention to convert them ALL to 2 tokens simultaneously. This tests the cumulative effect of tokenization changes across multiple entities and different types of vision-language tasks:

- **TallyQA**: Tests counting accuracy with modified entity tokenization
- **RefCOCO**: Tests spatial grounding with modified referring expression tokenization  
- **TextVQA**: Tests text reading comprehension with modified question tokenization

This comprehensive approach reveals whether tokenization effects are task-specific or universal across different VLM capabilities.


### Why can a word tokenize to one token or two in context?

- **Byte Pair Encoding (BPE)** and similar subword algorithms segment text into frequent chunks from a learned vocabulary.
- The same surface word can map to different token sequences depending on:
  - **Surrounding characters** (spaces, punctuation, quotes) that change pre-tokenization normalization and word boundaries.
  - **Casing or accents** that break merges.
  - **Whitespace handling**: some tokenizers learn merges that apply only when preceded by a space (e.g., Llama-style leading space rules).
- **Practically**: if a subword merge exists for "word" but only when preceded by a space, placing it within quotes, parentheses, or after special characters can disable that merge and split it into two tokens.

### This notebook's approach (ALL ENTITIES + MULTI-DATASET VERSION):

We enforce A vs B by applying minimal interventions around ALL eligible entities in each question/expression across multiple datasets:
- **Quotes**: `cat` and `dog` → `"cat"` and `"dog"` 
- **Parentheses**: `cat` and `dog` → `(cat)` and `(dog)`
- **Brackets**: `cat` and `dog` → `[cat]` and `[dog]`

For each intervention type, we identify ALL noun/adjective/adverb entities that tokenize as 1 token in the original text, then apply the intervention to convert them ALL to 2 tokens simultaneously. This tests the cumulative effect of tokenization changes across multiple entities and different types of vision-language tasks:

- **TallyQA**: Tests counting accuracy with modified entity tokenization
- **RefCOCO**: Tests spatial grounding with modified referring expression tokenization  
- **TextVQA**: Tests text reading comprehension with modified question tokenization

This comprehensive approach reveals whether tokenization effects are task-specific or universal across different VLM capabilities.
